# 02 — Grafo de conocimiento en Relate

Genera el Cypher de carga desde las 31 tripletas semilla, instala el servidor **Relate** (Neo4j, en memoria) dentro de la instancia de Colab, carga el grafo y corre las 12 preguntas en Cypher. Todo dentro de Colab: nada de pasos de máquina local.

**Prerrequisito:** notebook 01 (chunks + tripletas semilla).

In [ ]:
%%bash
nvidia-smi
python3 -c "import torch; print('torch', torch.__version__, '| CUDA disponible:', torch.cuda.is_available())" 2>/dev/null || echo "torch aun no instalado (se instala abajo)"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("Drive montado en /content/drive")

In [ ]:
# --- Parametros (editalos aqui arriba) ---
DRIVE_ROOT = "/content/drive/MyDrive/femsa-rag"   # carpeta en Drive donde vive todo
REPO_URL = "https://github.com/czammar/femsa-rag-graph-memo"
REPO = DRIVE_ROOT + "/femsa-rag-graph-memo"

In [ ]:
import os, subprocess

os.makedirs(DRIVE_ROOT, exist_ok=True)
# El cache de Hugging Face vive en Drive: los pesos descargados
# sobreviven al reinicio de la sesion (la VM de Colab es efimera).
os.environ["HF_HOME"] = DRIVE_ROOT + "/.cache/huggingface"
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

if os.path.isdir(REPO + "/.git"):
    print("repo ya existe, actualizando...")
    subprocess.run(["git", "-C", REPO, "pull", "--ff-only"], check=False)
else:
    print("clonando el repo...")
    subprocess.run(["git", "clone", REPO_URL, REPO], check=True)
print("repo en:", REPO)
print("HF_HOME:", os.environ["HF_HOME"])

## Dependencia: driver neo4j

La carga por Bolt usa el driver `neo4j` de Python: sin el paquete, la carga falla. Se instala aquí, antes de cargar el grafo.

In [ ]:
%pip install -q neo4j>=6.4.1

## Datos procesados: primero en Drive, si no, súbelos

Este notebook necesita `pipeline/data/chunks.jsonl` (las 151 páginas del informe extraídas). El flujo es:

1. **Buscarlo en Drive** (ruta estándar, bajo el repo clonado).
2. **Si no está, subirlo** con el widget de Colab de la celda siguiente: el archivo es `chunks.jsonl`. Lo consigues de una corrida anterior en Drive.
3. **Solo si no lo subes**, generarlo aquí mismo con la lógica de extracción inline (descarga el PDF ~19 MB y extrae las 151 páginas, ~1–2 min). Es el último recurso, no el camino principal. La lógica es la de `pipeline/extract.py` del repo, copiada en la celda: el notebook no ejecuta scripts del repo.

`triples_seed.json` y `eval_questions.json` vienen con el clon del repo: no hace falta subirlos.

In [ ]:
import json, subprocess, sys
from pathlib import Path

CHUNKS = REPO + "/pipeline/data/chunks.jsonl"
DATA_DIR = REPO + "/pipeline/data"


def _chunks_ok(path):
    try:
        with open(path) as f:
            first = f.readline()
            n = 1 + sum(1 for _ in f)
        row = json.loads(first)
        return n >= 100 and isinstance(row.get("text"), str) and len(row["text"]) > 0
    except Exception:
        return False


if _chunks_ok(CHUNKS):
    print(f"datos procesados OK en Drive: {CHUNKS}")
else:
    print("chunks.jsonl NO está en Drive.")
    print("Súbelo con el widget de abajo (botón 'Elegir archivos').")
    print("Qué subir: chunks.jsonl — las 151 páginas del informe FEMSA 2024.")
    print("Dónde conseguirlo: de una corrida anterior en Drive.")
    print("Si no lo subes, el notebook lo genera aquí mismo con la lógica inline (último recurso).")
    from google.colab import files
    uploaded = files.upload()
    for name, data in uploaded.items():
        dest = Path(DATA_DIR) / Path(name).name
        dest.parent.mkdir(parents=True, exist_ok=True)
        dest.write_bytes(data)
        print(f"subido: {dest} ({len(data)} bytes)")
    if not _chunks_ok(CHUNKS) and len(uploaded) == 1:
        # El archivo subido puede venir con otro nombre: probar como chunks.jsonl.
        only = next(iter(uploaded.values()))
        Path(CHUNKS).write_bytes(only)
        print("probando el archivo subido como chunks.jsonl...")
    if _chunks_ok(CHUNKS):
        print("datos listos por subida.")
    else:
        print("Sin subida válida: último recurso, generar los chunks aquí mismo...")
        print("(lógica de extracción inline: descarga el PDF ~19 MB y extrae las 151 páginas, ~1-2 min)")
        # El PDF está cifrado con AES (contraseña vacía): pypdf necesita el
        # paquete opcional `cryptography` para leerlo, que `pip install pypdf`
        # no trae. Se verifica aquí para que el último recurso funcione.
        try:
            import pypdf
            import cryptography
        except ImportError:
            subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                            "pypdf>=7.1.1", "cryptography>=3.1"], check=True)
            import pypdf
            import cryptography
        import urllib.request
        from pypdf import PdfReader

        # Lógica de extracción del informe, copiada inline: este notebook no
        # ejecuta scripts del repo. Solo la edición en inglés: el sitio en
        # español bloquea las descargas directas (WAF), así que la edición en
        # inglés es la que se descarga (mismo informe, mismas cifras).
        _PDF_URL_EN = "https://annualreport.femsa.com/pdf/FEMSA_2024_Annual_Report.pdf"
        _PDF_PATH_EN = Path(DATA_DIR) / "FEMSA_2024_Annual_Report_EN.pdf"
        _SOURCE_LABEL_EN = "FEMSA 2024 Integrated Annual Report, «Evolving» (English edition, same figures)"

        def _inline_is_valid_pdf(path):
            try:
                return len(PdfReader(str(path)).pages) > 0
            except Exception:
                return False

        def _inline_download_pdf():
            if _PDF_PATH_EN.exists() and _inline_is_valid_pdf(_PDF_PATH_EN):
                print(f"PDF en caché: {_PDF_PATH_EN} ({_PDF_PATH_EN.stat().st_size / 1e6:.1f} MB)")
                return _PDF_PATH_EN
            if _PDF_PATH_EN.exists():
                print(f"Caché inválido, se vuelve a descargar: {_PDF_PATH_EN}")
                _PDF_PATH_EN.unlink()
            Path(DATA_DIR).mkdir(parents=True, exist_ok=True)
            print(f"Descargando {_PDF_URL_EN} ...")
            req = urllib.request.Request(
                _PDF_URL_EN, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(req, timeout=300) as resp:
                with open(_PDF_PATH_EN, "wb") as fh:
                    fh.write(resp.read())
            if not _inline_is_valid_pdf(_PDF_PATH_EN):
                _PDF_PATH_EN.unlink()
                raise RuntimeError("El archivo descargado no es un PDF válido.")
            print(f"Guardado en {_PDF_PATH_EN} ({_PDF_PATH_EN.stat().st_size / 1e6:.1f} MB)")
            return _PDF_PATH_EN

        def _inline_extract_pages(pdf_path):
            reader = PdfReader(str(pdf_path))
            chunks = []
            for i, page in enumerate(reader.pages, start=1):
                text = (page.extract_text() or "").strip()
                chunks.append({"page": i, "text": text, "chars": len(text),
                               "fuente": _SOURCE_LABEL_EN})
            return chunks

        try:
            _pdf = _inline_download_pdf()
            _chunks = _inline_extract_pages(_pdf)
            _non_empty = sum(1 for c in _chunks if c["chars"] > 0)
            print(f"Páginas: {len(_chunks)} ({_non_empty} con texto)")
            Path(CHUNKS).parent.mkdir(parents=True, exist_ok=True)
            with open(CHUNKS, "w", encoding="utf-8") as fh:
                for c in _chunks:
                    fh.write(json.dumps(c, ensure_ascii=False) + "\n")
            print(f"{len(_chunks)} páginas -> {CHUNKS}")
        except Exception as e:
            print("Generación inline falló:", e)
        assert _chunks_ok(CHUNKS), "No se pudo obtener chunks.jsonl."
    print(f"chunks.jsonl listo: {CHUNKS}")

## Generar el Cypher de carga

La lógica que genera `load_graph.cypher` (MERGE idempotentes) y `schema.cypher` (9 constraints de unicidad) desde `triples_seed.json` va inline en la celda siguiente — la misma lógica que `pipeline/build_cypher.py`, sin ejecutar ese script. No se cargan tripletas extraídas sin revisar — esa regla sigue vigente aquí.

In [ ]:
import json

SEED = REPO + "/pipeline/triples_seed.json"
OUT_CYPHER = REPO + "/pipeline/load_graph.cypher"
OUT_SCHEMA = REPO + "/pipeline/schema.cypher"

# Ontología v1: relación en minúsculas -> tipo de relación en UPPER_SNAKE_CASE.
REL_MAP = {
    "tiene_division": "TIENE_DIVISION",
    "tiene_formato": "TIENE_FORMATO",
    "opera_en": "OPERA_EN",
    "reporta": "REPORTA",
    "mide": "MIDE",
    "crecio_en": "CRECIO_EN",
    "impulsa": "IMPULSA",
    "evoluciono_a": "EVOLUCIONO_A",
}

# Relaciones que llevan valor en la relación (el resto solo lleva fuente).
VALUE_RELATIONS = {"reporta", "mide"}

# Las 9 etiquetas de la ontología v1: (nombre corto del constraint, etiqueta).
# Ojo: IndicadorFinanciero -> `indicador_...`, MetricaOperativa -> `metrica_...`
# (no el nombre completo de la etiqueta en minúsculas).
LABELS = [("empresa", "Empresa"), ("division", "Division"),
          ("formato", "Formato"), ("indicador", "IndicadorFinanciero"),
          ("metrica", "MetricaOperativa"), ("periodo", "Periodo"),
          ("pais", "Pais"), ("iniciativa", "Iniciativa"),
          ("riesgo", "Riesgo")]


def cypher_str(value):
    # Escapa un string para usarlo como literal Cypher entre comillas simples.
    return "'" + value.replace("\\", "\\\\").replace("'", "\\'") + "'"


def cypher_value(value):
    if isinstance(value, bool):
        return "true" if value else "false"
    if isinstance(value, (int, float)):
        return repr(value)
    if value is None:
        return "null"
    return cypher_str(str(value))


def rel_props(t):
    # Propiedades de la relación para una tripleta.
    props = {}
    fuente = t.get("fuente", {}) or {}
    if fuente.get("documento"):
        props["fuente_documento"] = fuente["documento"]
    if fuente.get("pagina") is not None:
        props["fuente_pagina"] = fuente["pagina"]
    if t["relation"] in VALUE_RELATIONS:
        for key in ("valor", "unidad", "variacion_pct", "periodo",
                    "valor_exacto_informe"):
            if t.get(key) is not None:
                props[key] = t[key]
    if t.get("nota"):
        props["nota"] = t["nota"]
    return props


def props_cypher(props):
    if not props:
        return ""
    items = ", ".join(f"{k}: {cypher_value(v)}" for k, v in props.items())
    return " {" + items + "}"


def node_merge(label, name, var):
    return f"MERGE ({var}:{label} {{nombre: {cypher_str(name)}}})"


def triple_to_cypher(t, idx):
    # Una tripleta -> 1-2 sentencias MERGE (nodos + relación).
    #
    # Las variables llevan sufijo único por tripleta (s1/o1, s2/o2...):
    # Relate ejecuta un script completo con ámbito de variables compartido,
    # así que reutilizar `s`/`o` en cada sentencia falla con
    # VariableAlreadyBound. En Neo4j no haría falta, pero así el archivo
    # corre igual en ambos.
    rel = REL_MAP[t["relation"]]
    return [
        node_merge(t["subject_type"], t["subject"], f"s{idx}") + ";",
        node_merge(t["object_type"], t["object"], f"o{idx}") + ";",
        f"MERGE (s{idx})-[:{rel}{props_cypher(rel_props(t))}]->(o{idx});",
    ]


def derived_crecio_en(t, idx):
    # Deriva CRECIO_EN desde REPORTA/MIDE con variacion_pct.
    if t["relation"] not in VALUE_RELATIONS:
        return None
    if t.get("variacion_pct") is None or t.get("periodo") is None:
        return None
    fuente = t.get("fuente", {}) or {}
    props = {"variacion_pct": t["variacion_pct"]}
    if fuente.get("documento"):
        props["fuente_documento"] = fuente["documento"]
    if fuente.get("pagina") is not None:
        props["fuente_pagina"] = fuente["pagina"]
    return [
        node_merge(t["object_type"], t["object"], f"s{idx}") + ";",
        node_merge("Periodo", t["periodo"], f"o{idx}") + ";",
        f"MERGE (s{idx})-[:CRECIO_EN{props_cypher(props)}]->(o{idx});",
    ]


with open(SEED, encoding="utf-8") as fh:
    triples = list(json.load(fh)["triples"])

unknown = {t["relation"] for t in triples} - set(REL_MAP)
assert not unknown, f"Relaciones desconocidas en la ontología v1: {unknown}"

# Orden determinista para que el archivo generado sea estable.
triples.sort(key=lambda t: (t["relation"], t["subject"], t["object"]))

out = []
out.append("// load_graph.cypher — generado en este notebook desde triples_seed.json")
out.append("// Fuente: triples_seed.json (ontología v1, cifras verificadas)")
out.append("// Todo es MERGE: se puede correr varias veces sin duplicar.")
out.append("// Relaciones en UPPER_SNAKE_CASE (ontología: minúsculas).")
out.append("")

derived = []
seen_crecio = set()
for i, t in enumerate(triples, start=1):
    out.append(f"// ({t['subject']}, {t['relation']}, {t['object']})")
    out.extend(triple_to_cypher(t, i))
    d = derived_crecio_en(t, 1000 + i)
    if d:
        # Misma (indicador, variación, periodo, fuente) = misma relación:
        # p. ej. Proximidad Europa y Coca-Cola FEMSA, ambas +14.2% en
        # Ingresos Totales, colapsan en una sola CRECIO_EN.
        key = (t["object"], t.get("variacion_pct"), t.get("periodo"),
               (t.get("fuente") or {}).get("pagina"))
        if key not in seen_crecio:
            seen_crecio.add(key)
            derived.extend(d)
    out.append("")

if derived:
    out.append("// --- CRECIO_EN derivadas de REPORTA/MIDE (mismo hecho) ---")
    out.append("// Nota: si dos divisiones comparten indicador y variación")
    out.append("// (p. ej. Proximidad Europa y Coca-Cola FEMSA, ambas +14.2%")
    out.append("// en Ingresos Totales), MERGE las colapsa en UNA sola")
    out.append("// relación: es el mismo hecho a nivel de indicador.")
    out.append("")
    out.extend(derived)

with open(OUT_CYPHER, "w", encoding="utf-8") as fh:
    fh.write("\n".join(out))
n_nodes = len({(t["subject_type"], t["subject"]) for t in triples}
              | {(t["object_type"], t["object"]) for t in triples})
n_derived = sum(1 for line in out if "[:CRECIO_EN" in line)
print(f"[build] {len(triples)} tripletas -> {OUT_CYPHER}")
print(f"[build] Nodos distintos: {n_nodes}; "
      f"relaciones CRECIO_EN derivadas (distintas tras MERGE): {n_derived}")

# schema.cypher: una restricción de unicidad por etiqueta sobre `nombre`:
# es lo que hace que los MERGE de load_graph.cypher sean idempotentes
# sin duplicar nodos. OpenCypher estándar: corre en Neo4j 5+ y en Relate.
schema_lines = [
    "// schema.cypher — restricciones e índices para la ontología v1 (FEMSA 2024)",
    "//",
    "// OpenCypher estándar: corre en Neo4j 5+ y en Relate (relatedb.ai).",
    "// Una restricción de unicidad por etiqueta sobre `nombre`: es lo que hace",
    "// que los MERGE de load_graph.cypher sean idempotentes sin duplicar nodos.",
    "//",
    "// Nota sobre Relate: si tu versión de Relate aún no soporta CREATE",
    "// CONSTRAINT, la carga lo detecta, avisa y sigue (los MERGE",
    "// siguen funcionando; solo pierdes la garantía de unicidad a nivel de",
    "// base). En Neo4j las restricciones se crean sin problema.",
    "",
]
for short, label in LABELS:
    schema_lines.append(
        f"CREATE CONSTRAINT {short}_nombre_unique IF NOT EXISTS\n"
        f"FOR (n:{label}) REQUIRE n.nombre IS UNIQUE;\n")
with open(OUT_SCHEMA, "w", encoding="utf-8") as fh:
    fh.write("\n".join(schema_lines))
print(f"[build] esquema -> {OUT_SCHEMA} ({len(LABELS)} constraints)")

## Instalar Relate en la instancia

Relate publica binarios Linux x86_64 (`relate-linux-amd64`, versión fijada v1.2.0, la misma que se verificó en local). El instalador oficial (`curl -sSfL https://relatedb.ai/install.sh | bash`) resuelve el último release estable y verifica la firma; si el binario ya existe, no se reinstala. El binario queda en `~/.local/bin/relate` **dentro de la instancia** (se pierde al reiniciar la sesión: hay que re-correr esta celda).

In [ ]:
%%bash
if [ -x "$HOME/.local/bin/relate" ]; then
  echo "relate ya instalado: $($HOME/.local/bin/relate --version | head -1)"
else
  echo "instalando Relate..."
  curl -sSfL https://relatedb.ai/install.sh | bash
  "$HOME/.local/bin/relate" --version | head -1
fi
export PATH="$HOME/.local/bin:$PATH"
relate --version | head -1

## Arrancar el servidor Relate en segundo plano

El store `data/femsa.rl` vive en el repo (en Drive, así que persiste). El servidor escucha solo en localhost, sin autenticación (`--auth none`), igual que en local. Si la sesión de Colab se reinicia, hay que re-correr esta celda: el store no se pierde, solo el proceso.

In [ ]:
import os, subprocess, time

RELATE_BIN = os.path.expanduser("~/.local/bin/relate")
DB = REPO + "/pipeline/data/femsa.rl"
LOG = open("relate.log", "w")

def relate_up():
    try:
        import urllib.request
        with urllib.request.urlopen("http://localhost:7687", timeout=2) as r:
            return True
    except Exception:
        return False

# Bolt (7687) no responde a HTTP, asi que la prueba es: ¿el proceso vive y el puerto escucha?
import socket
def port_open(port):
    s = socket.socket()
    s.settimeout(2)
    try:
        s.connect(("127.0.0.1", port))
        return True
    except Exception:
        return False
    finally:
        s.close()

if port_open(7687):
    print("Relate ya escucha en el puerto 7687")
else:
    print("arrancando `relate serve` en segundo plano...")
    subprocess.Popen([RELATE_BIN, "serve", "--database", DB, "--bolt", "7687",
                      "--auth", "none"], stdout=LOG, stderr=subprocess.STDOUT,
                     preexec_fn=os.setsid)
    for _ in range(30):
        if port_open(7687):
            break
        time.sleep(2)
    print("Relate escucha en 7687:", port_open(7687))
    if not port_open(7687):
        print("Revisa relate.log")

## Cargar esquema + grafo por Bolt y verificar conteos

La lógica de carga va inline en la celda siguiente (driver `neo4j` de Python, Bolt a localhost:7687 — la misma que `pipeline/load_relate.py`, sin ejecutar ese script). Esperado, verificado: **27 nodos, 39 relaciones** (31 semilla + 8 `CRECIO_EN` derivadas). La carga es idempotente: correrla dos veces no duplica nada.

In [ ]:
import os
from neo4j import GraphDatabase
from neo4j.exceptions import AuthError, ServiceUnavailable

URI = os.environ.get("RELATE_URI", "bolt://localhost:7687")
SCHEMA_PATH = REPO + "/pipeline/schema.cypher"
LOAD_PATH = REPO + "/pipeline/load_graph.cypher"

START_RELATE_HINT = '''
No se pudo conectar a Relate. Para arrancar un servidor local:

    relate serve --database ./pipeline/data/femsa.rl --bolt 7687 --auth none

También puedes cargar el grafo sin servidor, directamente con el CLI:

    relate -d ./pipeline/data/femsa.rl query < schema.cypher
    relate -d ./pipeline/data/femsa.rl query < load_graph.cypher

Si tu servidor usa autenticación, define RELATE_USER y RELATE_PASSWORD.
'''


def split_statements(text):
    # Divide un archivo .cypher en sentencias por ';' (sin ';' en strings
    # en estos archivos: los textos con ';' no aparecen en los literales).
    out = []
    buf = []
    in_str = False
    i = 0
    while i < len(text):
        ch = text[i]
        if ch == "'":
            # Comilla escapada \' no cierra el string.
            if i > 0 and text[i - 1] == "\\":
                buf.append(ch)
            else:
                in_str = not in_str
                buf.append(ch)
        elif ch == ";" and not in_str:
            stmt = "".join(buf).strip()
            if stmt:
                out.append(stmt)
            buf = []
        else:
            buf.append(ch)
        i += 1
    tail = "".join(buf).strip()
    if tail:
        out.append(tail)
    return out


def read_statements(path):
    # Lee un .cypher ignorando líneas de comentario // y vacías.
    lines = []
    for line in open(path, encoding="utf-8").read().splitlines():
        if line.strip().startswith("//"):
            continue
        lines.append(line)
    return split_statements("\n".join(lines))


user = os.environ.get("RELATE_USER")
password = os.environ.get("RELATE_PASSWORD")
auth = (user, password) if user else None


def connect(auth_value):
    driver = GraphDatabase.driver(URI, auth=auth_value)
    driver.verify_connectivity()
    return driver


try:
    driver = connect(auth)
except AuthError:
    # El servidor pide autenticación y no se dio (o viceversa).
    fallback = ("neo4j", password or "neo4j") if not auth else None
    if fallback is None:
        raise SystemExit("[load] Autenticación rechazada con las credenciales "
                         "dadas." + START_RELATE_HINT)
    print("[load] Sin autenticación no funcionó; reintentando con "
          "usuario 'neo4j'...")
    try:
        driver = connect(fallback)
    except Exception:
        raise SystemExit("[load] Autenticación rechazada."
                         + START_RELATE_HINT)
except (ServiceUnavailable, Exception) as e:
    raise SystemExit(f"[load] No se pudo conectar a Relate en {URI}: {e}"
                     + START_RELATE_HINT)

print(f"[load] Conectado a {URI}")

schema_stmts = read_statements(SCHEMA_PATH)
load_stmts = read_statements(LOAD_PATH)
print(f"[load] schema.cypher: {len(schema_stmts)} sentencias; "
      f"load_graph.cypher: {len(load_stmts)} sentencias")

with driver.session() as session:
    # Esquema: si Relate no soporta CREATE CONSTRAINT, avisa y sigue.
    for stmt in schema_stmts:
        try:
            session.run(stmt)
        except Exception as e:
            print(f"[load] AVISO: no se pudo aplicar una sentencia de "
                  f"esquema ({e}). Sigo sin esa restricción; los MERGE "
                  f"siguen funcionando.")
            break
    print("[load] Esquema aplicado (o avisado).")
    for i, stmt in enumerate(load_stmts, start=1):
        session.run(stmt)
        if i % 25 == 0 or i == len(load_stmts):
            print(f"[load]   {i}/{len(load_stmts)} sentencias...")
    print("[load] Carga completa.")

    # Verificación.
    total_nodes = session.run("MATCH (n) RETURN count(n) AS c").single()["c"]
    total_rels = session.run("MATCH ()-[r]->() RETURN count(r) AS c").single()["c"]
    by_label = session.run(
        "MATCH (n) RETURN labels(n) AS etiqueta, count(n) AS n "
        "ORDER BY n DESC").data()
    by_rel = session.run(
        "MATCH ()-[r]->() RETURN type(r) AS tipo, count(r) AS n "
        "ORDER BY n DESC").data()
    q1 = session.run(
        "MATCH (e:Empresa {nombre: 'FEMSA'})-[r:REPORTA]->"
        "(i:IndicadorFinanciero {nombre: 'Ingresos Totales'}) "
        "RETURN r.valor AS valor, r.variacion_pct AS variacion").single()
    print(f"[load] Verificación: {total_nodes} nodos, {total_rels} relaciones.")
    print("[load] Nodos por etiqueta:")
    for row in by_label:
        print(f"[load]   {row['etiqueta']}: {row['n']}")
    print("[load] Relaciones por tipo:")
    for row in by_rel:
        print(f"[load]   {row['tipo']}: {row['n']}")
    print(f"[load] Prueba Q1: ingresos totales FEMSA 2024 = "
          f"Ps. {q1['valor']:,} M (+{q1['variacion']}%).")
    assert total_nodes == 27 and total_rels == 39, \
        f"Conteos inesperados: {total_nodes} nodos, {total_rels} relaciones (esperado 27/39)."
driver.close()

## Correr las 12 preguntas en Cypher

Con el CLI, sin pasar por el servidor: `relate -d <store> query` ejecuta cada consulta de `queries.cypher` contra el store. Separa el archivo por consultas (cada bloque `MATCH ... RETURN ...;`) y ejecuta una por una, imprimiendo la respuesta esperada al lado para comparar a ojo.

In [ ]:
import os, re, subprocess

RELATE_BIN = os.path.expanduser("~/.local/bin/relate")
DB = REPO + "/pipeline/data/femsa.rl"
text = open(REPO + "/pipeline/queries.cypher").read()

# Un bloque por pregunta: cada una empieza con su comentario `// Qn.`
# (la Q9 lleva DOS clausulas MATCH en una sola consulta: el regex toma
# del primer MATCH hasta el `;` final, asi que va junta).
blocks = re.split(r"(?=^// Q\d+\.)", text, flags=re.M)
n = 0
for b in blocks:
    m = re.search(r"MATCH.*?;", b, re.S)
    if not m:
        continue
    q = re.search(r"// (Q\d+\..+)", b)
    esp = re.search(r"Esperado: (.+)", b)
    p = subprocess.run([RELATE_BIN, "-d", DB, "query"], input=m.group(0).encode(),
                       capture_output=True)
    rows = [l for l in p.stdout.decode(errors="replace").splitlines() if l.startswith("{")]
    print((q.group(1).strip()[:90] if q else "consulta"), "| esperado:",
          (esp.group(1).strip() if esp else "?"))
    print("   ->", rows[0][:220] if rows else "(sin filas)")
    n += 1
print(f"\n{n} consultas ejecutadas (esperado: 12)")

## Alternativa sin servidor

Si el servidor no levanta, el grafo se puede cargar directamente con el CLI, sin Bolt (mismos archivos, mismo resultado):

```bash
relate -d ./pipeline/data/femsa.rl query < pipeline/schema.cypher
relate -d ./pipeline/data/femsa.rl query < pipeline/load_graph.cypher
```

desde el directorio del repo.

## Guardar en Drive (final)

Todo lo que produjo este notebook vive bajo el repo clonado en Drive (`REPO`), así que sobrevive aunque la sesión de Colab se corte. Esta celda lo verifica y lo lista: si algo falta, lo dice aquí, no después.

El store `femsa.rl` queda en Drive: el notebook 04 (MEMO) no lo necesita, pero cualquier corrida futura del grafo lo reusa sin recargar. Si la sesión se corta, basta re-correr las celdas de arranque del servidor.

In [ ]:
import os, shutil, time


def _human(n):
    for u in ["B", "KB", "MB", "GB"]:
        if n < 1024:
            return f"{n:.0f} {u}"
        n /= 1024
    return f"{n:.1f} GB"


def _list_dir(path):
    total, nfiles = 0, 0
    for root, _dirs, files in os.walk(path):
        for fn in sorted(files):
            p = os.path.join(root, fn)
            sz = os.path.getsize(p)
            total += sz
            nfiles += 1
            print(f"    {os.path.relpath(p, REPO)} ({_human(sz)})")
    return nfiles, total


# Logs de la sesion (viven en /content, fuera de Drive): copiarlos al repo.
for _log in ["ollama.log", "ollama_error.log", "relate.log"]:
    if os.path.isfile(_log):
        shutil.copy(_log, os.path.join(REPO, _log))
        print(f"log copiado a Drive: {_log}")

ENTRIES = [('pipeline/data/femsa.rl', 'Store Relate (27 nodos, 39 relaciones)', True), ('pipeline/load_graph.cypher', 'Cypher de carga generado (MERGE idempotentes)', True), ('pipeline/schema.cypher', 'Esquema (constraints)', False), ('relate.log', 'Log del servidor Relate', False)]
total_bytes, missing = 0, []
for rel, label, required in ENTRIES:
    p = os.path.join(REPO, rel)
    if os.path.isdir(p):
        nf, tb = _list_dir(p)
        total_bytes += tb
        print(f"{label}: directorio, {nf} archivos, {_human(tb)}")
    elif os.path.isfile(p):
        sz = os.path.getsize(p)
        total_bytes += sz
        mt = time.strftime("%Y-%m-%d %H:%M", time.localtime(os.path.getmtime(p)))
        print(f"{label}: {rel} ({_human(sz)}, {mt})")
    else:
        status = "FALTA (obligatorio)" if required else "pendiente (opcional)"
        print(f"{label}: {status}: {rel}")
        if required:
            missing.append(rel)
print(f"\nGUARDADO EN DRIVE: {_human(total_bytes)} verificados bajo {REPO}")
if missing:
    print("OJO, faltan obligatorios:", missing)